# DLIR data loading: starting point for the DL part

Loads the preprocessed HCP data, generates training slabs on the fly, wraps them in a PyTorch `DataLoader`, and scores
a prediction on the validation slabs exactly like the baselines.

**Data:** unpack `dlir_data.tar.gz` (Google Drive, see the README) into the repository root, or set the environment
variable `DLIR_DATA` to the unpacked `dlir_data` folder. **Packages:** numpy, scipy, nibabel, matplotlib; torch for the
`DataLoader` part.

**Conventions**
- All positions are scanner (world) millimetres, RAS, as given by the NIfTI affines.
- Fixed image: `data/<subject>/anat/T1.nii.gz` (1 mm). Moving image: a slab plus its affine (header); the header
  carries the start error, i.e. where registration starts.
- Target: `truth[i, j, k]` = the true T1 position (x, y, z mm) of slab voxel (i, j, k), distortion included.
  Loss / scoring mask: `brain & valid`.
- Error (TRE) of a slab: mean distance between predicted and true positions over the mask; success = TRE < 2 mm.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = Path(os.environ.get('DLIR_DATA', REPO / 'dlir_data'))
sys.path.insert(0, str(REPO / 'scripts'))                       # the slabsim package

from slabsim.dataset import Run, usable_runs
from slabsim.evaluation import displacement_error, fit_rigid, linear_estimate, rigid_errors
from slabsim.forward import sample
from slabsim.geometry import apply, voxel_centers
from slabsim.samples import load_sample

print('data:', DATA, '| subjects:', len(list((DATA / 'data').iterdir())))

## 1. Runs
Every run in `data/` passed the QC. The subject split is fixed in `splits/`; the full data give 770 training and
109 validation runs.

In [ ]:
split = lambda name: (REPO / 'splits' / f'hcp_{name}.txt').read_text().split()
train_runs = usable_runs(DATA / 'data', split('train'))
val_runs = usable_runs(DATA / 'data', split('val'))
print(len(train_runs), 'training runs,', len(val_runs), 'validation runs')
print(train_runs[0])
print(json.loads((train_runs[0] / 'acq.json').read_text()))

## 2. One run
`Run` loads the corrected whole-brain EPI and its field map and places both in T1 space by multiplying the anchor
into their headers (no resampling), plus the subject's T1 and brain mask. Each attribute is a `(data, affine)` pair.
Loading takes about 0.5 s, so reuse a `Run` for several slabs.

In [ ]:
run = Run(train_runs[0])
for name in ('t1', 'mask', 'epi', 'field'):
    data, affine = getattr(run, name)
    print(f'{name:6s} {str(data.shape):16s} voxel {np.round(np.linalg.norm(affine[:3, :3], axis=0), 2)} mm')

def on_t1_slice(img, k):
    """An image (data, affine) sampled on axial slice k of the T1 grid."""
    shape = run.t1[0].shape
    ijk = np.stack(np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), [k], indexing='ij'), -1)[:, :, 0]
    return sample(*img, apply(run.t1[1], ijk.astype(float)))

k = int(np.argwhere(run.mask[0] > 0)[:, 2].mean())             # an axial slice through the brain centre
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].imshow(run.t1[0][:, :, k].T, cmap='gray', origin='lower'); ax[0].set_title('T1 (fixed)')
ax[1].imshow(on_t1_slice(run.epi, k).T, cmap='gray', origin='lower'); ax[1].set_title('corrected EPI in T1 space')
im = ax[2].imshow(on_t1_slice(run.field, k).T, cmap='RdBu_r', vmin=-100, vmax=100, origin='lower')
ax[2].set_title('field map (Hz)'); fig.colorbar(im, ax=ax[2], fraction=0.04)
for a in ax: a.axis('off')
plt.show()

## 3. A training slab
`run.sample(rng)` simulates one random slab acquisition (geometry, phase encoding, distortion, head motion, start
error, appearance; ranges in the README) and returns:
- `image` the moving slab, `affine` its header (with the start error),
- `truth` the true T1 position of every voxel (mm), `brain` and `valid` (loss mask = both), `jacobian`,
- `params` the settings that were drawn.

In [ ]:
rng = np.random.default_rng(0)
s = run.sample(rng)
mask = s['brain'] & s['valid']
p = s['params']
print({k: p[k] for k in ('orientation', 'thickness', 'angles', 'voxel', 'pe_axis', 'pe_sign', 'trt', 'snr')})
print('slab', s['image'].shape, '| truth', s['truth'].shape, '| scored voxels', int(mask.sum()))

points = voxel_centers(s['affine'], s['image'].shape)           # where the header puts every voxel
start = np.linalg.norm(points - s['truth'], axis=-1)[mask].mean()
W = fit_rigid(points[mask], s['truth'][mask])                   # best rigid placement: what a rigid model can reach
floor = np.linalg.norm(apply(W, points) - s['truth'], axis=-1)[mask].mean()
print(f'start error {start:.1f} mm | best rigid fit {floor:.2f} mm (the rest is distortion)')

z = s['image'].shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].imshow(s['image'][:, :, z].T, cmap='gray', origin='lower', vmax=np.percentile(s['image'], 99))
ax[0].set_title('slab (moving), middle slice')
ax[1].imshow(sample(*run.t1, s['truth'][:, :, z]).T, cmap='gray', origin='lower'); ax[1].set_title('T1 at the true positions')
ax[2].imshow(mask[:, :, z].T, origin='lower'); ax[2].set_title('loss mask (brain & valid)')
for a in ax: a.axis('off')
plt.show()

## 4. PyTorch data loader
An endless stream of training slabs: each worker draws a run at random, loads it once and simulates several slabs
from it, with its own random generator. Slabs differ in size, so `batch_size=None` yields one slab at a time; for
batches, resample or crop the slabs to a fixed grid in your own collate function.

The T1 (179 x 224 x 224, float32) is sent with every slab here for simplicity; cache it per subject in the model
if that becomes a bottleneck.

In [ ]:
import torch
from torch.utils.data import DataLoader, IterableDataset, get_worker_info


class SlabStream(IterableDataset):
    """Random training slabs generated on the fly from a list of run folders."""

    def __init__(self, runs, seed=0, slabs_per_run=8):
        self.runs, self.seed, self.slabs_per_run = runs, seed, slabs_per_run

    def __iter__(self):
        info = get_worker_info()
        rng = np.random.default_rng([self.seed, info.id if info else 0])
        while True:
            run = Run(self.runs[rng.integers(len(self.runs))])
            for _ in range(self.slabs_per_run):
                s = run.sample(rng)
                p = s['params']
                yield {'slab': torch.from_numpy(s['image'][None]),            # (1, X, Y, Z)
                       'slab_affine': torch.from_numpy(s['affine']),          # (4, 4) voxel -> scanner mm (header)
                       't1': torch.from_numpy(run.t1[0][None]),               # (1, 179, 224, 224)
                       't1_affine': torch.from_numpy(run.t1[1]),
                       'truth': torch.from_numpy(s['truth']),                 # (X, Y, Z, 3) true T1 mm
                       'mask': torch.from_numpy(s['brain'] & s['valid']),     # (X, Y, Z) loss mask
                       'pe_axis': p['pe_axis'], 'pe_sign': p['pe_sign'],      # known acquisition metadata, same keys as
                       'trt': p['trt']}                                       # in the validation slabs' params.json


loader = DataLoader(SlabStream(train_runs), batch_size=None, num_workers=2)
t = time.time()
for i, batch in zip(range(6), loader):
    print(i, tuple(batch['slab'].shape), tuple(batch['truth'].shape), int(batch['mask'].sum()), 'masked voxels')
print(f'{(time.time() - t) / 6:.2f} s per slab with 2 workers (more workers -> faster)')

## 5. Validation slabs (`test/`)
Real distorted slices of the 30 validation subjects, 36 slabs per run, with the ground truth and the baseline
scores in `results.json`. Folder names: `t<thickness mm>_p<position %>_<start error>[_cor]`.

In [ ]:
slab_dirs = sorted(d for d in (DATA / 'test').glob('*/t*') if d.is_dir())
print(len(slab_dirs), 'validation slabs; first:', slab_dirs[0].relative_to(DATA))

d = slab_dirs[0]
image, affine, truth, mask = load_sample(d)
params = json.loads((d / 'params.json').read_text())
results = json.loads((d / 'results.json').read_text())
print('slab', image.shape, '| source run', params['run'], '| start error', round(params['initial_error_mm'], 1), 'mm')
print('phase encoding: axis', params['pe_axis'], '(in the slab grid), sign', params['pe_sign'], ', readout time', params['trt'], 's')
for method, r in sorted(results.items(), key=lambda x: x[1].get('mean', 1e9)):
    print(f"  {method:15s} TRE {r['mean']:6.2f} mm   rotation {r['rot_deg']:5.2f} deg   shift {r['trans_mm']:5.2f} mm")

## 6. Scoring a prediction
A model may predict a 4 x 4 transform (slab scanner mm -> T1 mm) or the T1 position of every slab voxel. `score`
computes the same numbers as the baselines; the checks reproduce two entries of `results.json`.

In [ ]:
def score(pred, slab_dir):
    """pred: (4, 4) world transform or (X, Y, Z, 3) T1 positions in mm. Returns TRE stats and rigid errors."""
    image, affine, truth, mask = load_sample(slab_dir)
    est = linear_estimate(pred, affine, image.shape) if np.shape(pred) == (4, 4) else pred
    out = {**displacement_error(est, truth, mask), **rigid_errors(est, truth, voxel_centers(affine, image.shape), mask)}
    out['success'] = out['mean'] < 2
    out['failure'] = out['rot_deg'] > 2 or out['trans_mm'] > 2
    return out

points = voxel_centers(affine, image.shape)
print('no registration:', round(score(np.eye(4), d)['mean'], 2), 'vs results.json', round(results['header']['mean'], 2))
oracle = fit_rigid(points[mask], truth[mask])
print('rigid oracle   :', round(score(oracle, d)['mean'], 2), 'vs results.json', round(results['rigid_oracle']['mean'], 2))

### The baseline to beat
AFNI on the validation slabs (full data: median 1.55 mm, 14 % failures, 109 s per slab).

In [ ]:
afni = [json.loads((d / 'results.json').read_text())['afni_lpc'] for d in slab_dirs]
tre = np.array([r['mean'] for r in afni])
fail = np.array([r['rot_deg'] > 2 or r['trans_mm'] > 2 for r in afni])
print(f'AFNI on {len(afni)} slabs: median TRE {np.median(tre):.2f} mm, failures {100 * fail.mean():.0f} %, '
      f'TRE < 2 mm on {100 * (tre < 2).mean():.0f} % of slabs')